# MOMENTUM 12-1 日频动量因子构造（训练集：2010-01-04 至 2020-12-31）

本 Notebook 只负责因子生产：

1. 从 Data/stocks_daily_04-25.parquet 读取复权价格和总市值；
2. 计算过去约12个月、跳过最近1个月的累计收益；
3. 要求动量起止价格有效、交易日位置准确，且窗口有效价格覆盖率至少80%；
4. 每日横截面做 MAD 去极值、对数总市值中性化和 Z-score；
5. 保存标准因子 Parquet，交给通用回测框架。

因子方向为正：历史赢家的因子值更高。计算只使用信号日及以前的信息，不构造未来收益、不计算IC、不进行分组回测。

方法参考 Kenneth French Data Library 的 Daily Momentum Factor：使用 prior (2-12) return，并要求约一年前和一个月前具有有效价格。

## 1. 导入依赖与冻结参数

日频近似使用252个交易日代表12个月，跳过最近21个交易日。程序会请求读取2008年末开始的预热数据；但当前行情文件最早为2010-01-04，因此严格模式不会回填未来信息，首批有效因子将自然从约2011年开始。

In [ ]:
from pathlib import Path
from IPython.display import display
import os

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:.6f}')

PROJECT_DIR = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'Back_test').exists() and (p / 'Factor').exists()), Path.cwd())
DATA_PATH = PROJECT_DIR / 'Data' / 'stocks_daily_04-25.parquet'
NOTEBOOK_DIR = PROJECT_DIR / 'Factor'

START_DATE = os.getenv('MOMENTUM_START_DATE', '2010-01-04')
END_DATE = os.getenv('MOMENTUM_END_DATE', '2020-12-31')
LOAD_START_DATE = os.getenv('MOMENTUM_LOAD_START_DATE', '2008-11-01')
OUTPUT_DIR = Path(os.getenv(
    'MOMENTUM_OUTPUT_DIR',
    str(NOTEBOOK_DIR / 'momentum_12_1_output_20100104_20201231')
))

LOOKBACK_TRADING_DAYS = 252
SKIP_RECENT_TRADING_DAYS = 21
REQUIRED_PRICE_COUNT = (
    LOOKBACK_TRADING_DAYS - SKIP_RECENT_TRADING_DAYS + 1
)
MIN_HISTORY_COVERAGE = 0.80
MIN_PRICE_COUNT = int(np.ceil(
    REQUIRED_PRICE_COUNT * MIN_HISTORY_COVERAGE
))
MAD_SCALE = 3.0
MIN_NEUTRALIZE_STOCKS = 100
EXCLUDE_ST = True

LOAD_START_TS = pd.Timestamp(LOAD_START_DATE)
START_TS = pd.Timestamp(START_DATE)
END_TS = pd.Timestamp(END_DATE)
LOAD_START_INT = int(LOAD_START_TS.strftime('%Y%m%d'))
END_INT = int(END_TS.strftime('%Y%m%d'))
DATE_TAG = f'{START_TS:%Y%m%d}_{END_TS:%Y%m%d}'

assert LOAD_START_TS < START_TS <= END_TS
assert LOOKBACK_TRADING_DAYS > SKIP_RECENT_TRADING_DAYS >= 1
assert DATA_PATH.exists(), f'找不到行情文件：{DATA_PATH}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('行情文件：', DATA_PATH)
print('请求的预热起点：', LOAD_START_DATE)
print('正式训练区间：', START_DATE, '至', END_DATE)
print('动量定义：T-252 至 T-21 的复权收益')
print('若源数据没有2010年前历史，首批有效值将约从2011年开始')
print('输出目录：', OUTPUT_DIR)
print('pandas / pyarrow：', pd.__version__, '/', pa.__version__)

## 2. 分批读取 Parquet

只读取股票代码、名称、日期、收盘价、复权因子和总市值，并在 Arrow 层过滤日期。不会载入2021年后的验证或测试数据。

In [ ]:
RAW_COLUMNS = [
    '股票代码', '股票名称', '交易日',
    '收盘价', '复权因子', '总市值（万元）'
]


def load_market_window(path, start_int, end_int, batch_size=300_000):
    parquet_file = pq.ParquetFile(path)
    selected = []
    scanned = 0

    for batch in parquet_file.iter_batches(
        batch_size=batch_size,
        columns=RAW_COLUMNS,
        use_threads=True,
    ):
        table = pa.Table.from_batches([batch])
        date_column = table['交易日']
        mask = pc.and_(
            pc.greater_equal(date_column, pa.scalar(start_int)),
            pc.less_equal(date_column, pa.scalar(end_int)),
        )
        filtered = table.filter(mask)
        if filtered.num_rows:
            selected.append(filtered)
        scanned += table.num_rows

    assert selected, '目标区间没有读取到数据'
    result = pa.concat_tables(selected).to_pandas()
    print(f'扫描 {scanned:,} 行，预热与训练区间保留 {len(result):,} 行')
    return result


market = load_market_window(DATA_PATH, LOAD_START_INT, END_INT)
market = market.rename(columns={
    '股票代码': 'code',
    '股票名称': 'name',
    '交易日': 'date_int',
    '收盘价': 'close',
    '复权因子': 'adj_factor',
    '总市值（万元）': 'total_market_cap',
})
market['date'] = pd.to_datetime(market.pop('date_int'), format='%Y%m%d')

market_dates = pd.Index(sorted(market['date'].dropna().unique()))
date_to_index = pd.Series(np.arange(len(market_dates)), index=market_dates)
market['date_index'] = market['date'].map(date_to_index)

# 去掉上市前等关键字段全部为空的占位行。
market = market.dropna(
    subset=['close', 'adj_factor', 'total_market_cap'], how='all'
)
market = market.sort_values(['code', 'date'], kind='stable').reset_index(drop=True)

assert market['date'].min() >= LOAD_START_TS
assert market['date'].max() <= END_TS
assert not market.duplicated(['date', 'code']).any(), '存在重复股票-日期记录'

print('实际载入日期：', market['date'].min().date(), '至', market['date'].max().date())
print('市场交易日：', f'{len(market_dates):,}')
print('载入行数：', f'{len(market):,}')
print('股票数量：', f"{market['code'].nunique():,}")
display(market.head())

## 3. 数据质量与复权价格

复权收盘价定义为 收盘价 × 复权因子，与现有回测框架的复权VWAP口径一致。价格、复权因子和总市值必须为正且有限；名称含ST的股票不进入当日因子横截面。

In [ ]:
for column in ['close', 'adj_factor', 'total_market_cap']:
    market[column] = pd.to_numeric(market[column], errors='coerce')

valid_price = (
    market['code'].notna()
    & np.isfinite(market['close'])
    & np.isfinite(market['adj_factor'])
    & (market['close'] > 0)
    & (market['adj_factor'] > 0)
)
valid_market_cap = (
    np.isfinite(market['total_market_cap'])
    & (market['total_market_cap'] > 0)
)
market['adjusted_close'] = (
    market['close'] * market['adj_factor']
).where(valid_price)
market['log_market_cap'] = np.log(
    market['total_market_cap'].where(valid_market_cap)
)

if EXCLUDE_ST:
    is_st = market['name'].fillna('').str.upper().str.contains('ST', regex=False)
else:
    is_st = pd.Series(False, index=market.index)

quality_summary = pd.Series({
    '载入行数': len(market),
    '有效复权价格行数': int(market['adjusted_close'].notna().sum()),
    '无效价格或复权因子行数': int((~valid_price).sum()),
    '有效总市值行数': int(valid_market_cap.sum()),
    'ST记录数': int(is_st.sum()),
    '重复股票日期数': int(market.duplicated(['date', 'code']).sum()),
})
display(quality_summary.to_frame('数量'))

## 4. 计算12-1动量原始值

公式：

MOM_12_1(T) = AdjustedClose(T-21) / AdjustedClose(T-252) - 1

这里不仅检查两个端点，还要求T-252至T-21之间共有232个有效复权价格，并确认股票记录与市场交易日序号连续。新股、长期停牌或历史不足的股票不会被勉强纳入。

In [ ]:
grouped_by_code = market.groupby('code', sort=False)
market['momentum_start_price'] = grouped_by_code['adjusted_close'].shift(
    LOOKBACK_TRADING_DAYS
)
market['momentum_end_price'] = grouped_by_code['adjusted_close'].shift(
    SKIP_RECENT_TRADING_DAYS
)

history_price_count = grouped_by_code['adjusted_close'].transform(
    lambda values: values.shift(SKIP_RECENT_TRADING_DAYS).rolling(
        REQUIRED_PRICE_COUNT,
        min_periods=MIN_PRICE_COUNT,
    ).count()
)
lookback_date_index = grouped_by_code['date_index'].shift(
    LOOKBACK_TRADING_DAYS
)
is_consecutive_history = (
    market['date_index'] - lookback_date_index
).eq(LOOKBACK_TRADING_DAYS)
has_complete_history = (
    history_price_count.ge(MIN_PRICE_COUNT)
    & is_consecutive_history
    & market['momentum_start_price'].notna()
    & market['momentum_end_price'].notna()
)

market['momentum_12_1_raw'] = (
    market['momentum_end_price'] / market['momentum_start_price'] - 1.0
)
valid_raw = (
    has_complete_history
    & np.isfinite(market['momentum_12_1_raw'])
    & (market['momentum_12_1_raw'] > -1.0)
)
market.loc[~valid_raw | is_st, 'momentum_12_1_raw'] = np.nan

display(
    market.loc[
        market['date'].ge(START_TS)
        & market['momentum_12_1_raw'].notna(),
        [
            'date', 'code', 'momentum_start_price',
            'momentum_end_price', 'momentum_12_1_raw'
        ],
    ].head()
)

## 5. 每日 MAD 去极值、市值中性化与标准化

每日横截面依次执行：

1. 中位数 ± 3 × MAD 去极值；
2. 回归 MOM_winsor = 截距 + beta × log(总市值) + 残差；
3. 对残差做 Z-score，得到 momentum_12_1_factor。

当前行情表没有完整行业与Barra风格字段，因此只进行Size中性化。最终因子已经预处理，回测时不能再次中性化。

In [ ]:
date_group = market.groupby('date', sort=False)['momentum_12_1_raw']
daily_median = date_group.transform('median')
absolute_deviation = (market['momentum_12_1_raw'] - daily_median).abs()
daily_mad = absolute_deviation.groupby(
    market['date'], sort=False
).transform('median')

lower_limit = daily_median - MAD_SCALE * daily_mad
upper_limit = daily_median + MAD_SCALE * daily_mad
clipped = market['momentum_12_1_raw'].clip(
    lower=lower_limit, upper=upper_limit
)
market['momentum_12_1_winsor'] = clipped.where(
    daily_mad > 0, market['momentum_12_1_raw']
)

eligible = (
    market['momentum_12_1_winsor'].notna()
    & market['log_market_cap'].notna()
)
daily_key = market['date']
x = market['log_market_cap'].where(eligible)
y = market['momentum_12_1_winsor'].where(eligible)
daily_n = eligible.groupby(daily_key, sort=False).transform('sum')
x_mean = x.groupby(daily_key, sort=False).transform('mean')
y_mean = y.groupby(daily_key, sort=False).transform('mean')
dx = x - x_mean
dy = y - y_mean
covariance_numerator = (dx * dy).groupby(
    daily_key, sort=False
).transform('sum')
x_variance_numerator = dx.pow(2).groupby(
    daily_key, sort=False
).transform('sum')
valid_regression = (
    (daily_n >= MIN_NEUTRALIZE_STOCKS)
    & (x_variance_numerator > 0)
)
size_beta = (covariance_numerator / x_variance_numerator).where(
    valid_regression
)
market['momentum_12_1_neutral'] = (
    dy - size_beta * dx
).where(valid_regression)

neutral_group = market.groupby(
    'date', sort=False
)['momentum_12_1_neutral']
neutral_mean = neutral_group.transform('mean')
neutral_std = neutral_group.transform('std')
market['momentum_12_1_factor'] = (
    (market['momentum_12_1_neutral'] - neutral_mean)
    / neutral_std.where(neutral_std > 0)
)

# 检查中性化后与log市值的相关性。
residual = market['momentum_12_1_neutral']
residual_centered = residual - residual.groupby(
    daily_key, sort=False
).transform('mean')
residual_covariance = (dx * residual_centered).groupby(
    daily_key, sort=False
).transform('sum')
residual_sum_of_squares = residual_centered.pow(2).groupby(
    daily_key, sort=False
).transform('sum')
neutralized_size_correlation = residual_covariance / np.sqrt(
    x_variance_numerator * residual_sum_of_squares
)
market['_neutralized_size_corr'] = neutralized_size_correlation

training_market = market['date'].between(START_TS, END_TS)
factor_coverage = market.loc[training_market].groupby('date').agg(
    market_rows=('code', 'size'),
    valid_adjusted_close=('adjusted_close', 'count'),
    complete_history=('momentum_12_1_raw', 'count'),
    neutralized_factor=('momentum_12_1_neutral', 'count'),
    valid_factor=('momentum_12_1_factor', 'count'),
    neutralized_size_correlation=('_neutralized_size_corr', 'first'),
)
factor_coverage['coverage'] = (
    factor_coverage['valid_factor'] / factor_coverage['market_rows']
)

valid_neutral_corr = factor_coverage[
    'neutralized_size_correlation'
].dropna()
assert not valid_neutral_corr.empty, '没有可用的市值中性化结果'
assert valid_neutral_corr.abs().max() < 1e-7, '市值中性化检查失败'
display(factor_coverage.describe().T)

## 6. 保存标准因子文件

输出只包含2010-01-04至2020-12-31。asof_date 等于信号日，表示每条记录只使用截至该日已知的数据。预热期和2021年后的数据都不会写入。

In [ ]:
factor_columns = [
    'date', 'code', 'name',
    'momentum_start_price', 'momentum_end_price',
    'momentum_12_1_raw', 'momentum_12_1_winsor',
    'momentum_12_1_neutral', 'momentum_12_1_factor',
]
factor_panel = market.loc[
    training_market & market['momentum_12_1_factor'].notna(),
    factor_columns,
].copy()
factor_panel['asof_date'] = factor_panel['date']

assert not factor_panel.empty
assert factor_panel['date'].min() >= START_TS
assert factor_panel['date'].max() <= END_TS
assert not factor_panel.duplicated(['date', 'code']).any()
assert np.isfinite(factor_panel['momentum_12_1_factor']).all()
assert (factor_panel['asof_date'] == factor_panel['date']).all()

factor_path = OUTPUT_DIR / f'momentum_12_1_factor_{DATE_TAG}.parquet'
factor_panel.to_parquet(factor_path, index=False, compression='zstd')
factor_coverage.to_csv(
    OUTPUT_DIR / 'factor_coverage.csv', index_label='date'
)

first_date = factor_panel['date'].min()
first_cross_section = factor_panel.loc[
    factor_panel['date'].eq(first_date), 'momentum_12_1_factor'
]
factor_summary = pd.Series({
    '因子记录数': len(factor_panel),
    '股票数量': factor_panel['code'].nunique(),
    '有效因子首日': first_date,
    '有效因子末日': factor_panel['date'].max(),
    '首日因子均值': first_cross_section.mean(),
    '首日因子标准差': first_cross_section.std(ddof=1),
    '中性化后最大绝对市值相关': valid_neutral_corr.abs().max(),
    '输出大小MB': factor_path.stat().st_size / 1024 / 1024,
})
display(factor_summary.to_frame('值'))
print('因子文件：', factor_path)
print('覆盖率与中性化诊断：', OUTPUT_DIR / 'factor_coverage.csv')
print('下一步：运行', PROJECT_DIR / 'Back_test' / '通用因子回测模板.ipynb')

## 7. 接入通用回测

配置：

    FACTOR_NAME = 'momentum_12_1'
    FACTOR_COLUMN = 'momentum_12_1_factor'
    FACTOR_DIRECTION = 'positive'
    FACTOR_PREPROCESSED = True

    FACTOR_SOURCE = (
        LEARNING_DIR / 'factor' / 'momentum_12_1_output_20100104_20201231'
        / 'momentum_12_1_factor_20100104_20201231.parquet'
    )

最终因子已经完成MAD、Size中性化和Z-score，因此回测中的 NEUTRALIZE 保持 False。训练回测仍应使用T+1 VWAP、非重叠持有期、交易成本和可交易性约束。

如果后续进入验证阶段，必须继续使用相同的252日回看、21日跳过和处理顺序，并使用训练期末历史数据预热；不能根据验证结果改成更好看的窗口。